# Wilson intervals for main metrics

This notebook computes the main metrics across each of the tasks, the current notebooks are all over the place so they are being moved.

In [1]:
import polars as pl
from weedvlm.evaluation.stats import wilson_table
df = pl.read_parquet("reasoning-results.parquet")

## Species identification

Split up on the question types for MCQ and OEQ questions

In [2]:
sid = df.filter(pl.col("task") == "species_id").with_columns(
    qt=pl.when(pl.col("question_type") == "species_id_multiple_choice").then(pl.lit("MCQ"))
    .when(pl.col("question_type") == "species_id_open_ended").then(pl.lit("OEQ"))
).filter(pl.col("qt").is_not_null())

wilson_table(sid, ["model", "qt"], pl.col("match"))

model,qt,n,k,rate,lo,hi
str,str,u32,u32,f64,f64,f64
"""anthropic/claude-haiku-5.5""","""MCQ""",1194,1036,0.867672,0.847267,0.885718
"""anthropic/claude-haiku-5.5""","""OEQ""",238,59,0.247899,0.197336,0.306472
"""deepseek/deepseek-v4.1-flash""","""MCQ""",1200,1074,0.895,0.876376,0.911103
"""deepseek/deepseek-v4.1-flash""","""OEQ""",240,75,0.3125,0.2572,0.373708
"""google/gemma-4-31b-it""","""MCQ""",1200,958,0.798333,0.774695,0.820068
"""google/gemma-4-31b-it""","""OEQ""",240,63,0.2625,0.210889,0.321595
"""openai/gpt-6-luna""","""MCQ""",1200,1028,0.856667,0.835701,0.875356
"""openai/gpt-6-luna""","""OEQ""",240,65,0.270833,0.218549,0.330339
"""openai/gpt-6.1-sol""","""MCQ""",640,605,0.9453125,0.924893,0.960418


## Fine-grained identification

The same process can be performed for fine-grained identification

In [3]:
fg = df.filter(pl.col("task") == "fine_grained")
wilson_table(fg, ["model"], pl.col("match"))

model,n,k,rate,lo,hi
str,u32,u32,f64,f64,f64
"""anthropic/claude-haiku-5.5""",400,275,0.6875,0.640474,0.730959
"""deepseek/deepseek-v4.1-flash""",400,297,0.7425,0.697484,0.782902
"""google/gemma-4-31b-it""",400,317,0.7925,0.750069,0.829366
"""openai/gpt-6-luna""",400,321,0.8025,0.760687,0.838558
"""openai/gpt-6.1-sol""",200,150,0.75,0.685658,0.804919


## Grounded weed VQA

Again for grounded weed VQA

In [4]:
gvqa = df.filter(pl.col("task") == "grounded_vqa")
wilson_table(gvqa, ["model"], pl.col("match"))

model,n,k,rate,lo,hi
str,u32,u32,f64,f64,f64
"""anthropic/claude-haiku-5.5""",300,284,0.946667,0.91513,0.966908
"""deepseek/deepseek-v4.1-flash""",299,280,0.936455,0.902891,0.958945
"""google/gemma-4-31b-it""",300,258,0.86,0.816168,0.894729
"""openai/gpt-6-luna""",300,273,0.91,0.872222,0.93741
"""openai/gpt-6.1-sol""",150,145,0.966667,0.924347,0.98568


## Weed localisation

Here we copy over the outcomes from weed localisation, and match on the 'correct' column

In [7]:
BOXES = pl.List(pl.List(pl.Float64))
loc = df.filter(pl.col("task") == "localisation")
loc = loc.with_columns(
    n_true=pl.col("boxes").str.json_decode(BOXES).list.len(),
    n_pred=pl.col("pred_boxes").str.json_decode(BOXES).list.len(),
).with_columns(
    loc_outcome=pl.when(pl.col("matched_iou").is_null()).then(pl.lit("miss"))
    .when(pl.col("matched_iou") < 0.5).then(pl.lit("poor_placement"))
    .when(pl.col("n_pred") > pl.col("n_true")).then(pl.lit("over_detect"))
    .when(pl.col("n_pred") < pl.col("n_true")).then(pl.lit("under_detect"))
    .otherwise(pl.lit("correct"))
)
loc = loc.with_columns(
    match=pl.col("loc_outcome") == "correct"
)
wilson_table(loc, ["model"], pl.col("match"))

model,n,k,rate,lo,hi
str,u32,u32,f64,f64,f64
"""anthropic/claude-haiku-5.5""",221,69,0.312217,0.254769,0.376083
"""deepseek/deepseek-v4.1-flash""",249,50,0.200803,0.155761,0.254937
"""google/gemma-4-31b-it""",250,94,0.376,0.318259,0.437494
"""openai/gpt-6-luna""",250,88,0.352,0.295444,0.413036
"""openai/gpt-6.1-sol""",123,31,0.252033,0.18361,0.335475


## Weed density estimation

Do the same for weed density estimation, split on annotated and unannotated

In [ ]:
wde = df.filter(pl.col("task") == "density").with_columns(
    qt=pl.when(pl.col("annotated") == 1).then(pl.lit("annotated"))
    .otherwise(pl.lit("unannotated"))
)
wilson_table(wde, ["model", "qt"], pl.col("match"))